# 模块一：数据治理与清洗管道 (Data Cleaning Pipeline)

**目标：**
本模块旨在通过 Pydantic 定义业务数据契约 (Schema)，并执行结构化的数据质量扫描。
我们将处理状态倒置、物流时间穿越、以及订单财务对账不一致等异常，最终输出数据质量血统表 (Data Lineage)。

In [13]:
# 1. 开启自动重载魔术命令（确保对 src 中 .py 文件的修改能即时生效）
%load_ext autoreload
%autoreload 2

import pandas as pd
import os
import sys

# 将上一级目录加入系统路径，以便导入 src 模块
sys.path.append('../')
from src.cleaner import OlistDataCleaner

# 忽略不必要的警告，保持输出整洁
import warnings
warnings.filterwarnings('ignore')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [14]:
# 2. 加载原始数据 (Raw Data)
print("Loading raw datasets...")
df_orders = pd.read_csv("../data/raw/olist_orders_dataset.csv")
df_payments = pd.read_csv("../data/raw/olist_order_payments_dataset.csv")
df_items = pd.read_csv("../data/raw/olist_order_items_dataset.csv")

print(f"Raw Orders: {len(df_orders):,} | Raw Payments: {len(df_payments):,}")

Loading raw datasets...
Raw Orders: 99,441 | Raw Payments: 103,886


## 核心管道执行与质量追踪 (Data Lineage)

调用 `OlistDataCleaner` 引擎执行 Schema 校验、业务逻辑检测与财务对账。

In [15]:
# 3. 实例化清洗引擎并运行管道
cleaner = OlistDataCleaner(df_orders, df_payments, df_items)

# 执行一键清洗
df_orders_clean, df_payments_clean, quality_report = cleaner.run_pipeline()

# 4. 渲染核心面试武器：数据质量血统表
print("✅ 清洗任务完成，数据血统追踪表如下：")
display(quality_report.get_lineage_df())

# 5. 保存干净的数据到 processed 文件夹，供后续模块使用
os.makedirs("../data/processed", exist_ok=True)
df_orders_clean.to_parquet("../data/processed/olist_orders_clean.parquet", index=False)
df_payments_clean.to_parquet("../data/processed/olist_payments_clean.parquet", index=False)
# 注意：为了让后续模块正常运行，这里也需要将 items 等宽表基础文件暂存一下
df_items.to_parquet("../data/processed/olist_items_clean.parquet", index=False)

✅ 清洗任务完成，数据血统追踪表如下：


,清洗步骤/检查项,输入记录数,淘汰记录数,淘汰率,输出记录数
0,Schema校验: 剔除越界与未知枚举值的支付数据,103886,11,0.01%,103875
1,业务扫描: 剔除状态倒置与时间线穿越订单,99441,1410,1.42%,98031
2,对账扫描: 剔除商品总金额与支付金额不符订单,98031,253,0.26%,97778
